In [18]:
import numpy as np
import pandas as pd
from numba import njit
import vectorbt as vbt
import pandas_ta as ta
import matplotlib.pyplot as plt
import seaborn as sns

from utils.KrakenHistoricalData import KrakenHistoricalData


In [19]:
# ========= 1. Data =========
interval = '4h'
# tickers = ["BTC-USD", "ETH-USD", "DOGE-USD","LTC-USD","XRP-USD"]
# data = vbt.YFData.download(tickers, start="2025-01-01", interval=interval)
# close = data.get("Close").astype(np.float64).copy(deep=True)
# high = data.get("High").astype(np.float64).copy(deep=True)
# low = data.get("Low").astype(np.float64).copy(deep=True)

#
tickers = ["BTC", "ETH", "LTC"]
k = KrakenHistoricalData()
end = pd.to_datetime("2025-09-30").tz_localize('UTC')
start = pd.to_datetime("2024-10-01").tz_localize('UTC')
data_df = k.get_ohlcv_df(tickers, interval=interval, start=start, end=end)

close = data_df.xs('close', axis=1, level=1)
high = data_df.xs('high', axis=1, level=1)
low = data_df.xs('low', axis=1, level=1)
open = data_df.xs('open', axis=1, level=1)
print(close.head())




symbol                              BTC          ETH        LTC
Datetime                                                       
2024-10-01 00:00:00+00:00  63679.199219  2628.139893  67.870003
2024-10-01 04:00:00+00:00  63999.898438  2644.780029  68.190002
2024-10-01 08:00:00+00:00  63720.199219  2628.830078  68.139999
2024-10-01 12:00:00+00:00  62624.101562  2523.639893  65.330002
2024-10-01 16:00:00+00:00  61695.398438  2486.489990  63.590000


In [20]:
# ========= 2. Signal Data Comparison =========

def entry_signals(close,
                  high,
                  low,
                  adx_length: int = 14,
                  adx_threshold: int = 25,
                  sar_acceleration: float = 0.02,
                  sar_maximum: float = 0.2):
    print(f"\npandas_ta SAR")
    sar_pandas_ta = vbt.pandas_ta('psar').run(high,
                                              low,
                                              close=close,
                                              acceleration=sar_acceleration,
                                              maximum=sar_maximum)
    print(sar_pandas_ta.psarl.tail())
    sar_buy = sar_pandas_ta.psarl_below(close)

    adx_pandas_ta = vbt.pandas_ta('adx').run(high, low, close, length=adx_length)
    dmp_cross = adx_pandas_ta.dmp_above(adx_pandas_ta.dmn)
    adx_threshold_cross = adx_pandas_ta.adx_above(adx_threshold)
    adx_buy = dmp_cross & adx_threshold_cross
    print(f"\ndmp > dmn")
    print(adx_buy.tail())

    return sar_buy & adx_buy


def stop_loss(close, high, low, atr_length: int = 14, atr_multiplier: float = 3.0, use_high: bool = False):
    atr = vbt.pandas_ta('atr').run(high, low, close, length=atr_length)
    print(atr.atrr.tail())
    atrr = atr.atrr

    if use_high:
        entry_price = high.where(entries).ffill()
    else:
        entry_price = close.where(entries).ffill()

    # 3) 3x ATR stop distance as a fraction of entry
    #    (this is already positive: 3 * atr below entry)
    print(atrr.info())
    print(entry_price.info())
    sl_stop = (atr_multiplier * atrr.values) / entry_price

    # Optional: if entry_price is NaN (no trade yet), keep NaN
    sl_stop = sl_stop.where(entry_price.notna())

    return sl_stop


entries = entry_signals(close, high, low, adx_length=14, adx_threshold=25, sar_acceleration=0.02, sar_maximum=0.2)
exits = pd.DataFrame(False, index=entries.index, columns=entries.columns)
sl_stop = stop_loss(close, high, low, atr_multiplier=3.0, atr_length=14)






pandas_ta SAR
symbol                               BTC          ETH         LTC
Datetime                                                         
2025-09-29 08:00:00+00:00  109381.096879  3912.824330  103.823492
2025-09-29 12:00:00+00:00  109621.401004  3922.321748  104.226273
2025-09-29 16:00:00+00:00  110085.150747  3939.455236  104.697194
2025-09-29 20:00:00+00:00  110602.800470  3955.560715  105.102187
2025-09-30 00:00:00+00:00  111136.284186  3978.129451  105.450480

dmp > dmn
adx_length                   14             
symbol                      BTC   ETH    LTC
Datetime                                    
2025-09-29 08:00:00+00:00  True  True  False
2025-09-29 12:00:00+00:00  True  True   True
2025-09-29 16:00:00+00:00  True  True   True
2025-09-29 20:00:00+00:00  True  True   True
2025-09-30 00:00:00+00:00  True  True   True
atr_length                         14                     
symbol                            BTC        ETH       LTC
Datetime                          

In [21]:

pf = vbt.Portfolio.from_signals(close,
                                entries=entries,
                                exits=exits,
                                size=1,
                                size_type='percent',
                                direction='longonly',
                                sl_stop=sl_stop,
                                sl_trail=True,
                                freq=interval,
                                fees=0.004,
                                init_cash=1000
                                )

In [22]:
pf_btc = pf[pf.wrapper.columns[0]]

stats_df = pd.DataFrame({col: pf[col].stats() for col in pf.wrapper.columns})

print(f'Ave Sharpe Ratio: {stats_df.loc["Sharpe Ratio"].mean():.2f}')
print(stats_df.tail().to_string())

# Stop price recorded at trade execution level
trade_records = pf.trades.records_readable

print(stats_df.info())
# stop from high --> 0.78  stop from close --> 0.82, stop from low --> 0.82

Ave Sharpe Ratio: 0.82
                      14                      
                     BTC        ETH        LTC
Expectancy     13.657953  62.912679 -11.466725
Sharpe Ratio    1.038598   1.676882  -0.261997
Calmar Ratio    1.518487   2.113386  -0.506682
Omega Ratio     1.098394    1.14958   0.975504
Sortino Ratio    1.55483   2.450097  -0.386646
<class 'pandas.core.frame.DataFrame'>
Index: 28 entries, Start to Sortino Ratio
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   (14, BTC)  28 non-null     object
 1   (14, ETH)  28 non-null     object
 2   (14, LTC)  28 non-null     object
dtypes: object(3)
memory usage: 1.9+ KB
None


In [23]:
def reassign_columns_value(stats_df: pd.DataFrame,trial_num, level: int = 0):
    new_level_0 = stats_df.columns.levels[level].to_numpy().copy()
    new_level_0[:] = trial_num
    new_levels = [pd.Index(new_level_0), stats_df.columns.levels[1]]
    new_columns = stats_df.columns.set_levels(new_levels)

    # Assign back
    stats_df.columns = new_columns

    stats_df.columns = stats_df.columns.set_names(['id', 'symbol'])
    return stats_df


def assign_column_labels(stats_df: pd.DataFrame, labels: list):
    return stats_df.rename_axis(labels, axis=1)


def stats_df_to_wide(df: pd.DataFrame, trial_num):
    df = reassign_columns_value(df,trial_num, level=0)
    df = assign_column_labels(df, labels=['id', 'symbol'])
    df = df.T.reset_index()
    return convert_cols_to_numeric(df)


def convert_cols_to_numeric(df: pd.DataFrame):
    df['Start'] = pd.to_datetime(df['Start'], errors='coerce')
    df['End'] = pd.to_datetime(df['End'], errors='coerce')
    df['Period'] = pd.to_timedelta(df['Period'], errors='coerce')
    duration_str = 'Duration'
    for col in df.columns:
        # Skip columns we already converted to datetime/timedelta
        if col in ['Start', 'End', 'Period']:
            continue

        # Skip columns that are already numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            continue
        if duration_str in col:
            df[col] = pd.to_timedelta(df[col], errors='coerce')
            continue
        # Try converting to numeric, catching exceptions
        try:
            df[col] = pd.to_numeric(df[col], errors='raise')
        except (ValueError, TypeError):
            # If conversion fails, leave the column as-is
            pass

    return df


num_levels = stats_df.columns.nlevels
print(f"\nNumber of levels: {num_levels}")  # e.g., 2

print(f"\n LONG")
print(stats_df.head())
print(stats_df.info())
wide = stats_df_to_wide(stats_df, trial_num=1)
print(f"\n WIDE")
print(wide)
print(wide.info())


Number of levels: 2

 LONG
                                    14                             \
                                   BTC                        ETH   
Start        2024-10-01 00:00:00+00:00  2024-10-01 00:00:00+00:00   
End          2025-09-30 00:00:00+00:00  2025-09-30 00:00:00+00:00   
Period               364 days 04:00:00          364 days 04:00:00   
Start Value                     1000.0                     1000.0   
End Value                  1321.895665                  1962.4684   

                                        
                                   LTC  
Start        2024-10-01 00:00:00+00:00  
End          2025-09-30 00:00:00+00:00  
Period               364 days 04:00:00  
Start Value                     1000.0  
End Value                   730.449734  
<class 'pandas.core.frame.DataFrame'>
Index: 28 entries, Start to Sortino Ratio
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   (14, B

In [24]:
# combine performance metrics with params
params = {'sar_acceleration': 0.02,
          'sar_maximum': 0.2,
          'atr_multiplier': 3,
          'adx_threshold': 0,  # has little affect
          'adx_length': 14,
          'ce_high_length': 22,
          'ce_low_length': 22,
          'atr_length': 14,  # atr multiplier for chandelier exit
          }

# dict to wide df
params_df = pd.DataFrame(params, index=wide.index)

print(params_df.head().to_string())

# combine
combined = pd.concat([wide, params_df], axis=1)
print(f"\n COMBINED")
print(combined.head())

print(f"\n LONG 2")
stats2 = stats_df.copy()
wide2 = stats_df_to_wide(stats2, trial_num=2)
combined2 = pd.concat([wide2, params_df], axis=1)
print(combined2.head())

print(f"\n COMBINED ALL")
all_combined = pd.concat([combined, combined2], axis=0).reset_index(drop=True)
print(all_combined.head(10))
print(all_combined.info())

   sar_acceleration  sar_maximum  atr_multiplier  adx_threshold  adx_length  ce_high_length  ce_low_length  atr_length
0              0.02          0.2               3              0          14              22             22          14
1              0.02          0.2               3              0          14              22             22          14
2              0.02          0.2               3              0          14              22             22          14

 COMBINED
   id symbol                     Start                       End  \
0   1    BTC 2024-10-01 00:00:00+00:00 2025-09-30 00:00:00+00:00   
1   1    ETH 2024-10-01 00:00:00+00:00 2025-09-30 00:00:00+00:00   
2   1    LTC 2024-10-01 00:00:00+00:00 2025-09-30 00:00:00+00:00   

             Period  Start Value    End Value  Total Return [%]  \
0 364 days 04:00:00       1000.0  1321.895665         32.189567   
1 364 days 04:00:00       1000.0  1962.468400         96.246840   
2 364 days 04:00:00       1000.0   730.

In [25]:
n = 1
pf[pf.wrapper.columns[n]].plot(width=1200, height=1900, title=f"{tickers[n]}").show()